In [ ]:
# Setup Environment, Verify GPU & Compile C++ Engine
# Kaggle notes: Do not pip-install torch; rely on Kaggle's preinstalled PyTorch.

import os
import torch

# Explicit GPU Check
print("GPU VERIFICATION:")
device = torch.device("cuda" if (torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 6) else "cpu")
if device.type == 'cuda':
    print(f"SUCCESS: Active GPU detected - {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No compatible GPU detected! Pipeline will run on CPU.")
print()

# Clone the repository if it's not already in the Kaggle working directory
repo_dir = "/kaggle/working/zerocross-ai"
if not os.path.exists(repo_dir):
    !git clone https://github.com/muhammad-hassaan-aiml/zerocross-ai.git {repo_dir}

os.chdir(repo_dir)

# Execute the comprehensive build and test script (includes Python smoke tests)
!chmod +x build_kaggle.sh
!./build_kaggle.sh


In [ ]:
# Auto-Resume Logic for Multi-Session Runs
import os
import shutil

# Kaggle automatically mounts your datasets in /kaggle/input/
input_dir = "/kaggle/input/"
working_models_dir = "/kaggle/working/models"

os.makedirs(working_models_dir, exist_ok=True)

# Search for previous run files and copy them to the working directory
found_previous = False
if os.path.exists(input_dir):
    for root, dirs, files in os.walk(input_dir):
        if "best_model.pth" in files:
            print(f"Found previous session data in {root}!")
            print("Copying to working directory to resume training...")
            for file in files:
                if file.endswith(".pth") or file.endswith(".pt") or file.endswith(".csv") or file.endswith(".json"):
                    source_path = os.path.join(root, file)
                    shutil.copy(source_path, working_models_dir)
            found_previous = True
            break

if not found_previous:
    print("No previous dataset attached. Starting a fresh Session #1.")
else:
    print("Resume data loaded successfully!")


In [ ]:
# Sanity Check Dry-Run
# Ensures the end-to-end pipeline works on Kaggle hardware before committing to a multi-hour session.
!python python/pipeline.py \
    --iterations 1 \
    --concurrent-games 5 \
    --games-per-iteration 5 \
    --mcts-sims 50 \
    --eval-games 2 \
    --eval-sims 20 \
    --batch-size 32 \
    --max-rejections 1


In [ ]:
# Run the AlphaZero Pipeline (Full Training)
!python python/pipeline.py \
    --iterations 50 \
    --concurrent-games 100 \
    --games-per-iteration 500 \
    --mcts-sims 200 \
    --eval-games 40 \
    --eval-sims 200 \
    --batch-size 512 \
    --max-rejections 5 \
    --num-res-blocks 6 \
    --num-channels 128
